# 09C · What defines the raw PCA clusters?
Inspect the completed 09B cell clusters from the Ada-14R tile. A rare population, a low-RNA cell and a cell at a compartment boundary can occupy distinct regions of the PCA graph. Use the cluster profiles, observed markers and tissue locations together when deciding which groups merit follow-up.

Run beside 09B using the same scientific Python kernel. This CPU analysis reads the saved PCA scores, assignments and the prepared count H5AD. It rebuilds one neighbor graph and compares Leiden resolutions. Tables use the **evaluation core**. The graph includes the same eligible core and halo cells used in 09B.

In [ ]:
from pathlib import Path
import io
import json
import math
import os
import tempfile
import zipfile
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.metrics import adjusted_rand_score
from IPython.display import display, FileLink

RESULT_ROOT = Path("09_pilot_results")
RUN_DIR = None  # Optional explicit path to a completed 09B_<id> directory.
SOURCE_H5AD = None  # Set only if you moved the prepared all_cells.h5ad.
CHECK_RESOLUTIONS = (0.2, 0.4)

## Open the completed run and align cells

In [ ]:
if RUN_DIR is None:
    candidates = sorted(p.parent for p in RESULT_ROOT.glob("09B_*/complete.json"))
    preferred = RESULT_ROOT / "09B_4403598b0ac3"
    if preferred in candidates:
        run = preferred
    elif len(candidates) == 1:
        run = candidates[0]
    else:
        raise ValueError(f"Set RUN_DIR to a completed 09B directory. Found {candidates}")
else:
    run = Path(RUN_DIR).expanduser()
run = run.resolve()
receipt = json.loads((run / "complete.json").read_text())
settings = json.loads((run / "settings.json").read_text())
recorded = Path(settings["source"])
source = Path(SOURCE_H5AD).expanduser() if SOURCE_H5AD else recorded
if not source.is_file() and SOURCE_H5AD is None:
    source = run.parent / "inputs" / recorded.parent.name / recorded.name
if source.stat().st_size != settings["size"]:
    raise ValueError("The prepared H5AD size differs from the 09B source receipt.")
if source.stat().st_mtime_ns != settings["mtime_ns"]:
    print("Source timestamp differs from the receipt. Confirm that this is a copy of the 09B input.")

local = pd.read_csv(run / "local_cell_assignments.csv.gz", dtype=str, keep_default_na=False)
core_labels = local["evaluation_core"].str.lower()
if not core_labels.isin(["true","false"]).all():
    raise ValueError("Unrecognized evaluation_core flag.")
core = core_labels.eq("true").to_numpy()
z = np.load(run / "raw_PCA_latent.npy", allow_pickle=False)
if (not local.cell_id.is_unique or z.shape != (len(local), settings["settings"]["latent_dim"])
        or len(local) != receipt["fitted_cells"] or core.sum() != receipt["evaluation_core_cells"]
        or not np.isfinite(z).all()):
    raise ValueError("The PCA rows or the assignment counts disagree with the 09B receipt.")
core_df = local.loc[core].reset_index(drop=True).copy()
cl = core_df.raw_PCA_leiden.rename("PCA_cluster")
print(run.name, "•", len(local), "fitted cells •", int(core.sum()), "evaluation-core cells")

## Profile the observed counts

Full-assay UMIs, detected genes and the fraction of counts from the most abundant gene describe each cell's RNA content. Panel support is the fraction of cells with at least two **distinct** detected panel genes. The panel table lists the available genes. The mixed-panel fraction requires independent pan-T and myeloid support in the same cell. Mixed signals can reflect local tissue composition, assignment error or a biological state. A second table gives direct coreceptor detection and mixed support specifically among cells previously labeled T in each cluster.

For the marker table, rank all assayed genes by the difference in detection frequency between one PCA cluster and the other core cells. Use the frequencies and cell counts to judge small groups. These descriptive contrasts use one region of one mouse.

In [ ]:
panels = {
    "panT": "Cd3d Cd3e Cd3g Trac Trbc1 Trbc2",
    "myeloid": "C1qa C1qb C1qc Csf1r Adgre1 Lyz2",
    "B": "Cd79a Cd79b Ms4a1 Cd19",
    "endothelial": "Cdh5 Pecam1 Kdr Emcn",
    "stromal": "Col1a1 Col1a2 Dcn Lum",
    "epithelial": "Epcam Krt8 Krt18 Krt19",
}
data = ad.read_h5ad(source, backed="r")
try:
    genes = pd.Index(data.var_names)
    ids = data.obs_names.get_indexer(local.cell_id)
    if (not genes.is_unique or (ids < 0).any() or (np.diff(ids) <= 0).any()
            or data.n_obs != receipt["source_cells"]):
        raise ValueError("The cell IDs or original 09B source-row order do not match the H5AD.")
    obs = data.obs.iloc[ids]
    for key in ["cell_type", "section_id"]:
        if not np.array_equal(obs[key].astype(str).to_numpy(), local[key].to_numpy()):
            raise ValueError(f"{key} differs between the prepared H5AD and saved assignments.")
    xy = np.asarray(data.obsm["spatial"])[ids[core], :2] * settings["settings"]["microns_per_unit"]
    rows = ids[core]
    # Read contiguous source windows once; keep only the selected core rows.
    blocks = []
    for group in np.split(rows, np.flatnonzero(np.diff(rows // 2048)) + 1):
        lo, hi = group[0], group[-1] + 1
        blocks.append(data.X[lo:hi][group - lo].tocsr())
    counts = sparse.vstack(blocks, format="csr")
finally:
    data.file.close()
del blocks
counts.sum_duplicates()
counts.eliminate_zeros()
det = counts.copy()
det.data = np.ones(det.nnz, dtype=np.int32)
qc = pd.DataFrame(index=core_df.index)
qc["full_UMIs"] = np.asarray(counts.sum(axis=1)).ravel()
qc["detected_genes"] = det.getnnz(axis=1)
qc["top_gene_share"] = counts.max(axis=1).toarray().ravel() / np.maximum(qc["full_UMIs"], 1)
# Decile ties can make the overall fraction exceed 10%.
depth_cut = float(qc.full_UMIs.quantile(0.1))
qc["low_depth"] = qc.full_UMIs.le(depth_cut)
panel_rows = []
for panel, requested in panels.items():
    measured = [g for g in requested.split() if g in genes]
    panel_rows.append((panel, len(measured), ", ".join(measured)))
    qc[panel] = (np.asarray(det[:, genes.get_indexer(measured)].sum(axis=1)).ravel() >= 2
                 if len(measured) >= 2 else np.nan)
qc["panT_and_myeloid"] = ((qc.panT & qc.myeloid) if qc[["panT","myeloid"]].notna().all().all()
                         else np.nan)
observed = lambda gene: np.asarray(det[:, genes.get_loc(gene)].toarray()).ravel() > 0
cd4 = observed("Cd4")
cd8 = observed("Cd8a") | observed("Cd8b1")
is_T = core_df.cell_type.str.contains(receipt["type_patterns"]["T"], regex=True, case=False, na=False)
T_evidence = pd.DataFrame({"CD4_only":cd4 & ~cd8, "CD8_only":cd8 & ~cd4, "both":cd4 & cd8,
                           "neither":~(cd4 | cd8), "panT_and_myeloid":qc["panT_and_myeloid"]})
T_profile = T_evidence.loc[is_T].groupby(cl[is_T]).mean().add_suffix("_fraction")
T_profile.insert(0, "n_preannotated_T",cl[is_T].value_counts())
T_profile = T_profile.reset_index()
panel_coverage = pd.DataFrame(panel_rows, columns=["panel","n_assayed","assayed_genes"])
g = qc.groupby(cl)
profile = g[["full_UMIs","detected_genes","top_gene_share"]].median().add_prefix("median_")
profile["n_core"] = g.size()
profile["fraction_of_core"] = profile.n_core / len(cl)
profile["low_depth_fraction"] = g.low_depth.mean()
support = g[[*panels, "panT_and_myeloid"]].mean().add_suffix("_fraction")
profile = profile.join(support).sort_values("n_core", ascending=False)
def cross_table(partition, labels, name):
    tab = pd.crosstab(partition, labels).stack().rename("n").reset_index()
    tab.columns = ["PCA_cluster", name, "n"]
    tab = tab.loc[tab.n > 0].copy()
    tab["fraction_of_PCA_cluster"] = tab.n / tab.groupby("PCA_cluster").n.transform("sum")
    return tab.sort_values(["PCA_cluster","n"], ascending=[True,False])
composition = cross_table(cl, core_df.cell_type, "previous_cell_type")
other_clusters = []
for other in ("scVI_leiden","ResolVI_leiden"):
    if other in core_df:
        t = cross_table(cl,core_df[other],"other_cluster")
        t["method"] = other.removesuffix("_leiden")
        other_clusters.append(t)
model_overlap = pd.concat(other_clusters,ignore_index=True)
major = composition.drop_duplicates("PCA_cluster").set_index("PCA_cluster")
profile["most_frequent_previous_label"] = major.previous_cell_type
profile["fraction_with_that_label"] = major.fraction_of_PCA_cluster
labels = pd.Categorical(cl, categories=profile.index)
membership = sparse.csr_matrix((np.ones(len(cl), dtype=np.int32), (labels.codes, np.arange(len(cl)))),
                               shape=(len(profile),len(cl)))
n_positive = (membership @ det).toarray()
total_positive = n_positive.sum(axis=0)
markers = []
for key, pos in zip(profile.index, n_positive):
    n = int(profile.at[key,"n_core"])
    if n == len(cl):
        continue
    frac_in = pos / n
    frac_out = (total_positive - pos) / (len(cl) - n)
    delta = frac_in - frac_out
    for j in np.argsort(-delta, kind="stable")[:10]:
        if delta[j] > 0:
            markers.append((key,genes[j],int(pos[j]),n,frac_in[j],frac_out[j],delta[j]))
markers = pd.DataFrame(markers, columns=["PCA_cluster","gene","cells_detected","n_core",
                                         "fraction_in","fraction_out","difference"])
print("Core 10th-percentile depth:", depth_cut, "UMIs")
display(profile[["n_core","median_full_UMIs","median_detected_genes","median_top_gene_share",
                 "low_depth_fraction","most_frequent_previous_label","fraction_with_that_label"]].round(3))
display(T_profile.round(3))
display(panel_coverage)

## Compare Leiden resolutions on the saved PCA graph

Rebuild neighbors on the saved PCA coordinates for all fitted cells using the 09B settings. The table reports agreement with the original labels on the core (adjusted Rand index, 1 means identical partitions up to label names). The rerun at the original resolution checks graph reconstruction in this environment. The transition counts show which original groups combine or split.

In [ ]:
seed = int(settings["settings"]["seed"])
original_res = float(settings["settings"]["resolution"])
graph = ad.AnnData(X=sparse.csr_matrix((len(z), 1), dtype=np.float32))
graph.obsm["latent"] = np.ascontiguousarray(z, dtype=np.float32)
sc.pp.neighbors(graph, n_neighbors=min(20, len(z)-1), use_rep="latent", random_state=seed)
res_rows, transitions = [], []
for resolution in sorted(set([*CHECK_RESOLUTIONS, original_res])):
    sc.tl.leiden(graph, resolution=float(resolution), random_state=seed, key_added="audit_labels",
                 flavor="igraph", n_iterations=2, directed=False)
    candidate = graph.obs["audit_labels"].astype(str).to_numpy()[core]
    ari = adjusted_rand_score(cl, candidate)
    res_rows.append((resolution, len(np.unique(candidate)), ari))
    tab = cross_table(cl, candidate, "candidate_cluster")
    tab["resolution"] = resolution
    transitions.append(tab)
resolution_table = pd.DataFrame(res_rows, columns=["resolution","n_core_clusters","ARI_to_saved_PCA"])
transitions = pd.concat(transitions, ignore_index=True)
display(resolution_table.round(3))

## Inspect the clusters in tissue and return the audit

In [ ]:
def draw_facets(section):
    keep = core_df.section_id.eq(section).to_numpy()
    order = [k for k in profile.index if (keep & cl.eq(k).to_numpy()).any()]
    fig, axes = plt.subplots(math.ceil(len(order)/4),4, figsize=(11,2.35*math.ceil(len(order)/4)),
                             squeeze=False, constrained_layout=True)
    for ax in axes.flat:
        ax.set_axis_off()
    for ax, key in zip(axes.flat, order):
        member = keep & cl.eq(key).to_numpy()
        ax.scatter(*xy[keep].T, s=1, color="#d9dee1", linewidths=0, rasterized=True)
        ax.scatter(*xy[member].T, s=3, color="#07665d", linewidths=0, rasterized=True)
        ax.set_aspect("equal")
        ax.set_title(f"PCA {key}  ·  {member.sum():,} cells", fontsize=10)
    fig.suptitle(f"{section}  |  Evaluation core", fontsize=14)
    return fig

def panel_figure():
    width = max(7, len(panels) * 1.2)
    fig, ax = plt.subplots(figsize=(width,max(2.8, 1.5 + .26*len(profile))), constrained_layout=True)
    mat = profile[[p + "_fraction" for p in panels]].to_numpy(dtype=float)
    c = ax.imshow(mat, cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(np.arange(len(panels)), labels=list(panels))
    ax.set_yticks(np.arange(len(profile)), labels=[f"{k}  (n={v:,})" for k,v in profile.n_core.items()])
    ax.set_ylabel("Saved PCA cluster")
    ax.set_title("Observed panel support in the evaluation core")
    fig.colorbar(c, ax=ax, label="Fraction with ≥2 distinct panel genes")
    return fig

meta = {"source_run":run.name, "fitted_cells":len(local), "core_cells":len(cl),
        "saved_core_clusters":len(profile), "core_low_depth_threshold_UMIs":depth_cut,
        "threshold_definition":"core 10th percentile, inclusive", "seed":seed,
        "original_resolution":original_res, "resolutions":resolution_table.resolution.tolist(),
        "spatial_sections":core_df.section_id.unique().tolist()}
tables = {"cluster_profiles.csv":profile.reset_index(), "panel_coverage.csv":panel_coverage,
          "previous_T_evidence.csv":T_profile, "model_cluster_overlap.csv":model_overlap,
          "top_detection_markers.csv":markers, "cluster_label_composition.csv":composition,
          "resolution_summary.csv":resolution_table, "resolution_transitions.csv":transitions}
# Publish only a finished ZIP. Per-cell inputs and outputs stay out of the archive.
archive = Path.cwd() / f"09C_return_to_chat_{run.name.removeprefix('09B_')}.zip"
with tempfile.TemporaryDirectory(dir=archive.parent) as tmp:
    staged = Path(tmp) / archive.name
    with zipfile.ZipFile(staged, "w", compression=zipfile.ZIP_DEFLATED) as pack:
        for name, table in tables.items():
            pack.writestr(name,table.to_csv(index=False))
        pack.writestr("audit_settings.json",json.dumps(meta,indent=2))
        def save_view(fig, name):
            with io.BytesIO() as buf:
                fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
                pack.writestr(name, buf.getvalue())
            display(fig)
            plt.close(fig)
        save_view(panel_figure(), "marker_support.png")
        for number, section in enumerate(meta["spatial_sections"], 1):
            save_view(draw_facets(section), f"spatial_core_section_{number}.png")
    os.replace(staged,archive)
display(FileLink(archive.name))

To review a small cluster, start with its core cell count, median UMIs, most frequently detected distinguishing genes and the fraction with panel support. Check whether its members occupy one tissue region and where they go at lower resolution. A small group with consistent markers and coherent tissue localization warrants a different interpretation from scattered, low-depth cells dominated by one transcript. These diagnostics describe the cluster and leave the cell annotations intact.